In [ ]:
# 02_feature_engineering.ipynb

# -------------------------------
# Step 1: Import libraries
# -------------------------------
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.preprocessing import OneHotEncoder

# Set paths
INPUT_PATH = Path("../data/cleaned_data.csv")
OUTPUT_PATH = Path("../data/engineered_data.csv")

# -------------------------------
# Step 2: Load dataset
# -------------------------------
df = pd.read_csv(INPUT_PATH)
print("Shape before feature engineering:", df.shape)
df.head()

# -------------------------------
# Step 3: Create new features
# -------------------------------
# Premium to income ratio
df["premium_to_income"] = df["premium"] / (df["Income"] + 1)

# Ratio of premiums paid to age (years)
df["premiums_paid_ratio"] = df["no_of_premiums_paid"] / (df["age_in_years"] + 1)

# Total late counts
df["total_late_counts"] = (
    df["Count_3-6_months_late"] 
    + df["Count_6-12_months_late"] 
    + df["Count_more_than_12_months_late"]
)

# Flag: High underwriting score (>= 99th percentile)
threshold = df["application_underwriting_score"].quantile(0.99)
df["high_underwriting_flag"] = (df["application_underwriting_score"] >= threshold).astype(int)

# -------------------------------
# Step 4: Encode categorical variables
# -------------------------------
categorical_cols = ["sourcing_channel", "residence_area_type"]

encoder = OneHotEncoder(drop="first", sparse_output=False)
encoded = encoder.fit_transform(df[categorical_cols])

encoded_df = pd.DataFrame(
    encoded, 
    columns=encoder.get_feature_names_out(categorical_cols),
    index=df.index
)

# Drop original categorical columns and join encoded
df = df.drop(columns=categorical_cols).join(encoded_df)

# -------------------------------
# Step 5: Reorder columns
# -------------------------------
cols = [c for c in df.columns if c != "renewal"] + ["renewal"]
df = df[cols]

# -------------------------------
# Step 6: Save engineered dataset
# -------------------------------
df.to_csv(OUTPUT_PATH, index=False)
print("✅ Engineered dataset saved to", OUTPUT_PATH)
print("Shape after feature engineering:", df.shape)
df.head()


Shape before feature engineering: (79853, 13)
Index(['sourcing_channel_B', 'sourcing_channel_C', 'sourcing_channel_D',
       'sourcing_channel_E', 'residence_area_type_Urban'],
      dtype='object')
✅ Engineered dataset saved to ..\data\engineered_data.csv
Shape after feature engineering: (79853, 20)


,perc_premium_paid_by_cash_credit,age_in_days,Income,Count_3-6_months_late,Count_6-12_months_late,Count_more_than_12_months_late,application_underwriting_score,no_of_premiums_paid,premium,age_in_years,premium_to_income,premiums_paid_ratio,total_late_counts,high_underwriting_flag,sourcing_channel_B,sourcing_channel_C,sourcing_channel_D,sourcing_channel_E,residence_area_type_Urban,renewal
0,0.429,12058.0,355060.0,0.0,0.0,0.0,99.02,13.0,3300.0,33,0.009294,0.382353,0.0,0,0.0,1.0,0.0,0.0,1.0,1
1,0.010,21546.0,315150.0,0.0,0.0,0.0,99.89,21.0,18000.0,59,0.057115,0.350000,0.0,1,0.0,0.0,0.0,0.0,1.0,1
2,0.917,17531.0,84140.0,2.0,3.0,1.0,98.69,7.0,3300.0,48,0.039220,0.142857,6.0,0,0.0,1.0,0.0,0.0,0.0,0
3,0.049,15341.0,250510.0,0.0,0.0,0.0,99.57,9.0,9600.0,42,0.038322,0.209302,0.0,0,0.0,0.0,0.0,0.0,1.0,1
4,0.052,31400.0,198680.0,0.0,0.0,0.0,99.87,12.0,9600.0,86,0.048319,0.137931,0.0,0,1.0,0.0,0.0,0.0,1.0,1
